# FusionIntell — Situational Room (Colab launcher)

Runs the whole pipeline in this Colab session and loads the dashboard
inline below — no local install needed.

**Before you start:** you'll need your own `ANTHROPIC_API_KEY`,
`CESIUM_TOKEN`, `ADSBX_KEY`, and `DATALASTIC_KEY`. Keys you type below are
kept only in this ephemeral Colab VM — they are never saved into this
notebook file.

Run the cells in order (Runtime → Run all).

In [ ]:
import os

REPO_URL = "https://github.com/aimanaltoubi/situational-room.git"
REPO_DIR = "situational-room"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL}
%cd {REPO_DIR}

!apt-get -qq update && apt-get -qq install -y default-jre-headless
!pip install -q -r requirements.txt

## Enter your API keys
Input is hidden (`getpass`). Leave a field blank to skip a key — the
matching feature (e.g. weekly AI report) just won't work.

In [ ]:
from getpass import getpass

keys = {
    "ANTHROPIC_API_KEY": "Claude report generation",
    "CESIUM_TOKEN":      "3D globe rendering",
    "ADSBX_KEY":         "ADS-B flights + GPS jamming",
    "DATALASTIC_KEY":    "Marine vessel tracking",
}

with open(".env", "w") as f:
    for key, purpose in keys.items():
        value = getpass(f"{key} ({purpose}): ")
        f.write(f"{key}={value}\n")

print("\n.env written for this session.")

## Build the dashboard data
`skip_scrape=True` uses cached flight history instead of the slow live
scraper — uncheck it for a full, slower refresh.

In [ ]:
skip_scrape = True  #@param {type:"boolean"}

!python3 verify_setup.py

flag = "--skip-scrape" if skip_scrape else ""
!python3 run_pipeline.py {flag}

## Load System
Displays the built dashboard inline, the same as clicking **Load System**
on the app's local main page.

In [ ]:
from IPython.display import IFrame, display, HTML

html_path = "output/ifs_globe.html"
if os.path.exists(html_path):
    display(HTML("<h3>System built — dashboard below</h3>"))
    display(IFrame(src=html_path, width=1200, height=800))
else:
    display(HTML("<h3>Dashboard not built — re-run the pipeline cell above</h3>"))

## Weekly AI report (optional)
Requires `ANTHROPIC_API_KEY` to have been set above.

In [ ]:
import glob

reports = sorted(glob.glob("output/weekly_prediction_*.html"), reverse=True)
if reports:
    display(IFrame(src=reports[0], width=1200, height=800))
else:
    display(HTML("<h3>No weekly report found</h3>"))